In [2]:
import pandas as pd
import xml.etree.ElementTree as ET

def rss_to_dataframe(source):
    if source.startswith("http://") or source.startswith("https://"):
        import urllib.request
        with urllib.request.urlopen(source) as response:
            tree = ET.parse(response)
    else:
        tree = ET.parse(source)

    root = tree.getroot()
    
    channel = root.find("channel")
    items = channel.findall("item")

    records = []
    for item in items:
        record = {}
        for child in item:
            # Strip namespace prefix if present: {http://...}tagname → tagname
            tag = child.tag.split("}")[-1] if "}" in child.tag else child.tag
            record[tag] = child.text
        records.append(record)

    return pd.DataFrame(records)

df = rss_to_dataframe("https://news.google.com/rss/search?q=okc&hl=en-US&gl=US&ceid=US%3Aen")

print(df.shape)
print(df.columns.tolist())
print(df.head())

(102, 6)
['title', 'link', 'guid', 'pubDate', 'description', 'source']
                                               title  \
0  Thunder 131-122 Suns (Apr 27, 2026) Final Scor...   
1  Oklahoma City Thunder vs Phoenix Suns Apr 27, ...   
2  Midtown OKC to get giant cake sculpture, see r...   
3  4 takeaways: Thunder sweep Suns out of the pla...   
4  Dozens of OKC companies seek employees of all ...   

                                                link  \
0  https://news.google.com/rss/articles/CBMibkFVX...   
1  https://news.google.com/rss/articles/CBMiZ0FVX...   
2  https://news.google.com/rss/articles/CBMi5wFBV...   
3  https://news.google.com/rss/articles/CBMid0FVX...   
4  https://news.google.com/rss/articles/CBMikAFBV...   

                                                guid  \
0  CBMibkFVX3lxTFA0a1ZQaE16VVhMT0h2bnRkWVcyOFJfZ0...   
1  CBMiZ0FVX3lxTE1XS2lvYUZBQzRaOWN6cWN1SXJmeUVzOT...   
2  CBMi5wFBVV95cUxOMjdHX04tVFhuRmsyTkswMlRWZkgtZ0...   
3  CBMid0FVX3lxTE14aUNydEpNaE91

In [3]:
df = df[['title', 'pubDate']]
print(df.head())

                                               title  \
0  Thunder 131-122 Suns (Apr 27, 2026) Final Scor...   
1  Oklahoma City Thunder vs Phoenix Suns Apr 27, ...   
2  Midtown OKC to get giant cake sculpture, see r...   
3  4 takeaways: Thunder sweep Suns out of the pla...   
4  Dozens of OKC companies seek employees of all ...   

                         pubDate  
0  Tue, 28 Apr 2026 19:15:29 GMT  
1  Tue, 28 Apr 2026 02:05:10 GMT  
2  Tue, 28 Apr 2026 19:53:00 GMT  
3  Tue, 28 Apr 2026 13:06:29 GMT  
4  Mon, 27 Apr 2026 21:58:19 GMT  


In [4]:
#from transformers import pipeline

#sentiment = pipeline(
#    "sentiment-analysis",
#    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
#)

#def get_sentiment(text):
#    result = sentiment(text, truncation=True, max_length=512)[0]
#    return result['label'], result['score']

#df[['sentiment', 'sentiment_score']] = df['title'].apply(
#    lambda x: pd.Series(get_sentiment(x))
#)

#print(df[['title', 'sentiment', 'sentiment_score']].head())

In [5]:
df['pubDate'] = pd.to_datetime(df['pubDate']).dt.strftime('%Y-%m-%d %H:%M:%S')

In [6]:
final_df = df[['pubDate', 'title']]
print(final_df)

                 pubDate                                              title
0    2026-04-28 19:15:29  Thunder 131-122 Suns (Apr 27, 2026) Final Scor...
1    2026-04-28 02:05:10  Oklahoma City Thunder vs Phoenix Suns Apr 27, ...
2    2026-04-28 19:53:00  Midtown OKC to get giant cake sculpture, see r...
3    2026-04-28 13:06:29  4 takeaways: Thunder sweep Suns out of the pla...
4    2026-04-27 21:58:19  Dozens of OKC companies seek employees of all ...
..                   ...                                                ...
97   2026-04-26 22:52:17  Victor Wembanyama, Spurs suffocate Blazers in ...
98   2026-04-25 04:37:29  4 takeaways: Lakers-Rockets Game 3 leaves Hous...
99   2026-04-21 08:26:26  Timberwolves-Nuggets Game 2: Wolves claw back ...
100  2026-04-24 05:48:59  4 takeaways: Cavaliers-Raptors Game 3 ends in ...
101  2026-04-18 07:00:00  Rockets-Lakers Game 1: Luke Kennard steps up f...

[102 rows x 2 columns]


In [7]:
final_df.to_csv('okc_headlines_data.csv', index=False)